# 02 - Receipt pipeline on Colab (GPU)
**Preprocess → CRAFT → VietOCR → LayoutXLM features** for MC-OCR 2021.

All logic lives in `src/` (unit-tested). This notebook only installs dependencies, builds the
pipeline from `configs/` and runs it. Runtime → *Change runtime type* → **GPU**.

Order: install → load config → **smoke test (5 images)** → full run → visualise.

## 1. Get the code

In [ ]:
import os, sys
if not os.path.exists("QuetHoaDon"):
    !git clone https://github.com/CaoLePhucThinh2203/QuetHoaDon.git
%cd QuetHoaDon
sys.path.insert(0, os.getcwd())

## 2. Install dependencies (restart the runtime if Colab asks)

In [ ]:
!pip install -q craft-text-detector --no-deps
!pip install -q vietocr --no-deps
!pip install -q gdown einops prefetch_generator albumentations lmdb imgaug
!pip install -q transformers sentencepiece kagglehub pyyaml

In [ ]:
# detectron2 is needed by LayoutXLM's visual backbone (source build, several minutes)
!pip install -q ninja
!pip install 'git+https://github.com/facebookresearch/detectron2.git' --no-build-isolation

## 3. Dataset

In [ ]:
import kagglehub
DATASET_DIR = kagglehub.dataset_download("domixi1989/vietnamese-receipts-mc-ocr-2021")
print(DATASET_DIR)

## 4. Build the pipeline from `configs/`

In [ ]:
import glob, json, warnings
import cv2, torch
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=UserWarning)

from src.common.config import load_config
from src.preprocessing.image_preprocessor import ImagePreprocessor
from src.detection.craft_detector import CraftDetector
from src.recognition.vietocr_recognizer import VietOCRRecognizer
from src.features.layoutxlm_extractor import LayoutXLMFeatureExtractor
from src.pipeline.runner import OutputDirs, ReceiptPipeline

cfg = load_config("configs")
M = cfg["model"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = cfg["runtime"]["paths"]["colab"]["output_dir"]
print("device:", DEVICE, "| output:", OUTPUT_DIR)

In [ ]:
dirs = OutputDirs.from_base(OUTPUT_DIR)
pipeline = ReceiptPipeline(
    preprocessor=ImagePreprocessor(**M["preprocessing"]),
    detector=CraftDetector(M["detection"], device=DEVICE),
    recognizer=VietOCRRecognizer(M["recognition"], device=DEVICE, batch_size=M["recognition"]["batch_size"]),
    extractor=LayoutXLMFeatureExtractor(M["layout_encoder"]["model_name"], device=DEVICE,
                                        max_tokens=M["layout_encoder"]["max_seq_length"],
                                        save_fp16=M["layout_encoder"]["save_fp16"]),
    dirs=dirs,
)

## 5. Select images
`SPLIT = None` processes every image. To process one split, first run
`python scripts/prepare_dataset.py --csv ... --images ... --out outputs/dataset` and set `SPLIT`.

In [ ]:
SPLIT = None   # None | "train" | "validation" | "test"

images = []
for ext in ("jpg", "JPG", "jpeg", "png"):
    images += glob.glob(os.path.join(DATASET_DIR, "**", f"*.{ext}"), recursive=True)
images = sorted(images)

if SPLIT:
    import csv
    with open("outputs/dataset/mcocr_manifest_split.csv", encoding="utf-8") as f:
        wanted = {r["img_id"] for r in csv.DictReader(f) if r["split"] == SPLIT}
    images = [p for p in images if os.path.basename(p) in wanted]

print(f"{len(images)} images (split={SPLIT})")

## 6. Smoke test (5 images) - run this first

In [ ]:
stats, failed = pipeline.run(images[:5], abort_after=5)

## 7. Full run (resumes automatically after a disconnect)

In [ ]:
stats, failed = pipeline.run(images)

## 8. Inspect one result

In [ ]:
def visualize(file_id):
    json_path, pt_path, img_path = dirs.paths(file_id)
    meta = json.load(open(json_path, encoding="utf-8"))
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    for k, it in enumerate(meta["ocr_items"]):
        x0, y0, x1, y1 = it["bbox"]
        cv2.rectangle(img, (x0, y0), (x1, y1), (255, 0, 0), 1)
        cv2.putText(img, str(k), (x0, max(10, y0 - 2)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)
    plt.figure(figsize=(8, 12)); plt.imshow(img); plt.axis("off"); plt.show()
    print(f"skew={meta.get('skew_angle', float('nan')):.2f}  regions={meta['n_text_regions']}  "
          f"truncated={meta.get('possibly_truncated')}  time={meta.get('timing_sec', {}).get('total')}s")
    for k, it in enumerate(meta["ocr_items"][:40]):
        print(f"{k:>3}  {it['ocr_confidence']:.2f}  {it['text']}")

ids = [os.path.splitext(os.path.basename(p))[0] for p in images]
visualize(next(i for i in ids if os.path.exists(dirs.paths(i)[0])))